In [1]:
from dataset.cell_motion import CellMotionRollout
from model.simulator import Simulator
import torch
from torch_geometric.loader import DataLoader
from torch_geometric.data import Data
import torch_geometric.transforms as T
import numpy as np
import os
from typing import Optional
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from scipy.spatial import cKDTree, Delaunay

# 设置计算设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# 设置路径
load_dir = 'checkpoint/20260204_161356/best_epoch/best_checkpoint.pth'
dataset_dir = os.path.join(os.getcwd(), 'dataset')

# 训练/推理参数
batch_size = 1
sequence_length = 20  # 历史帧数
max_rollout_steps = 100  # 默认 rollout 步数
time_step = 1.0  # 时间步长（与训练时一致）

# 网络参数（应与训练时一致）
hidden_size = 256
message_passing_num = 15
node_input_size = sequence_length * 2 + 1  # e.g. 20帧 * 2维 + 1时间
edge_input_size = 3
learning_rate = 1.811923871799656e-04

# 创建模拟器并加载 checkpoint
simulator = Simulator(
    message_passing_num=message_passing_num,
    node_input_size=node_input_size,
    edge_input_size=edge_input_size,
    device=device,
    hidden_size=hidden_size,
    model_dir=load_dir
)

try:
    loaded = simulator.load_checkpoint()  # 期望它设置 simulator.model 并返回模型（或 None）
except TypeError:
    # 如果 load_checkpoint 需要参数（例如路径），使用给定路径
    loaded = simulator.load_checkpoint(load_dir)

# 确保模型存在并处于 eval 模式
if hasattr(simulator, 'model') and simulator.model is not None:
    model = simulator.model
else:
    # 如果 load_checkpoint 返回模型对象则用它
    model = loaded
if model is None:
    raise RuntimeError("无法加载模型：请检查 Simulator.load_checkpoint API 与提供的路径。")
model.to(device)
model.eval()


# transforms（计算边特征）
transformer = T.Compose([
    T.Cartesian(norm=False),
    T.Distance(norm=False)
])

Using device: cuda
Simulator model initialized with hidden_size=256
Simulator model loaded checkpoint checkpoint/20260204_161356/best_epoch/best_checkpoint.pth


In [2]:
def build_edges_from_geometry(pos, original_edge_attr=None, method='knn', k=6, 
                               radius=None, self_loop=False, device='cpu'):
    """
    基于节点几何位置构建邻接关系，并生成与原始维度匹配的边属性
    
    Args:
        pos: (N, 2) 节点位置坐标 (torch.Tensor 或 np.ndarray)
        original_edge_attr: 原始边属性 (用于获取维度信息)
        method: 邻接构建方式，可选 'knn', 'radius', 'delaunay'
        k: KNN 方法中的邻域数量
        radius: 半径方法中的距离阈值
        self_loop: 是否包含自环
        device: 张量所在设备
    
    Returns:
        edge_index: (2, E) LongTensor，边索引
        edge_attr: (E, D) FloatTensor，边属性（维度与输入一致）
    """
    
    # 确保输入是 numpy 数组
    if isinstance(pos, torch.Tensor):
        pos_np = pos.cpu().detach().numpy()
    else:
        pos_np = np.array(pos)
    
    # 获取原始边属性的维度
    if original_edge_attr is not None:
        if isinstance(original_edge_attr, torch.Tensor):
            edge_attr_dim = original_edge_attr.shape[1] if original_edge_attr.dim() > 1 else 1
        else:
            edge_attr_dim = original_edge_attr. shape[1] if len(original_edge_attr. shape) > 1 else 1
    else:
        edge_attr_dim = 2  # 默认维度：距离 + 幅值
    
    N = pos_np.shape[0]
    
    if method == 'knn':
        edge_index, distances = _build_knn_edges(pos_np, k, self_loop)
    elif method == 'radius':
        if radius is None:
            radius = np.mean(np.linalg.norm(pos_np - pos_np. mean(axis=0), axis=1)) * 0.3
        edge_index, distances = _build_radius_edges(pos_np, radius, self_loop)
    elif method == 'delaunay':
        edge_index, distances = _build_delaunay_edges(pos_np, self_loop)
    else:
        raise ValueError(f"未知的构建方法: {method}")
    
    # 根据距离生成边属性，匹配维度
    if edge_index.shape[1] > 0:
        edge_attr = _generate_edge_attributes(
            pos_np, edge_index, distances, edge_attr_dim
        )
    else:
        edge_attr = np.zeros((0, edge_attr_dim), dtype=np.float32)
    
    # 转换为 torch 张量
    edge_index = torch.tensor(edge_index, dtype=torch.long, device=device)
    edge_attr = torch.tensor(edge_attr, dtype=torch.float32, device=device)
    
    return edge_index, edge_attr


def _build_knn_edges(pos_np, k, self_loop=False):
    """KNN 方法构建边"""
    N = pos_np.shape[0]
    k = min(k, N - 1)
    
    tree = cKDTree(pos_np)
    distances, indices = tree.query(pos_np, k=k + 1)
    
    edge_index_list = []
    distances_list = []
    
    for i in range(N):
        neighbors = indices[i, 1:]
        dists = distances[i, 1:]
        
        for j, neighbor in enumerate(neighbors):
            if neighbor < N:
                edge_index_list.append([i, neighbor])
                distances_list.append(dists[j])
    
    edge_index = np.array(edge_index_list). T if edge_index_list else np. zeros((2, 0), dtype=np.long)
    distances = np.array(distances_list)
    
    return edge_index, distances


def _build_delaunay_edges(pos_np, self_loop=False):
    """基于 Delaunay 三角剖分构建边"""
    N = pos_np.shape[0]
    
    if N < 3:
        return _build_knn_edges(pos_np, k=min(2, N-1), self_loop=self_loop)
    
    try:
        tri = Delaunay(pos_np)
    except Exception as e:
        return _build_knn_edges(pos_np, k=min(6, N-1), self_loop=self_loop)
    
    edge_set = set()
    for simplex in tri.simplices:
        for i in range(3):
            for j in range(i + 1, 3):
                edge = tuple(sorted([simplex[i], simplex[j]]))
                edge_set.add(edge)
    
    edge_index_list = []
    distances_list = []
    
    for i, j in edge_set:
        for src, tgt in [(i, j), (j, i)]:
            dist = np.linalg.norm(pos_np[tgt] - pos_np[src])
            edge_index_list.append([src, tgt])
            distances_list.append(dist)
    
    edge_index = np.array(edge_index_list).T if edge_index_list else np.zeros((2, 0), dtype=np.long)
    distances = np.array(distances_list)
    
    return edge_index, distances


def _generate_edge_attributes(pos_np, edge_index, distances, dim):
    """
    根据几何信息生成指定维度的边属性
    
    Args:
        pos_np: (N, 2) 节点位置
        edge_index: (2, E) 边索引
        distances: (E,) 距离数组
        dim: 目标维度
    
    Returns:
        edge_attr: (E, dim) 边属性
    """
    E = edge_index.shape[1]
    edge_attr = np.zeros((E, dim), dtype=np.float32)
    
    if E == 0:
        return edge_attr
    
    # 第 0 列：距离
    edge_attr[:, 0] = distances
    
    if dim >= 2:
        # 第 1 列：方向向量的 x 分量
        directions = pos_np[edge_index[1]] - pos_np[edge_index[0]]
        edge_attr[:, 1] = directions[:, 0]
    
    if dim >= 3:
        # 第 2 列：方向向量的 y 分量
        edge_attr[:, 2] = directions[:, 1]
    
    if dim >= 4:
        # 第 3 列：距离的平方
        edge_attr[:, 3] = distances ** 2
    
    if dim >= 5:
        # 第 4 列：向量的幅值（应该等于距离，作为冗余特征）
        edge_attr[:, 4] = np.linalg.norm(directions, axis=1)
    
    # 如果维度更高，填充相同的特征或零
    if dim > 5:
        edge_attr[:, 5:] = 0.0
        
    return edge_attr

In [3]:
def compute_velocity_features(graph_x, sequence_len):
    """
    辅助函数：将绝对位置特征转换为速度（差分）特征。
    假设 graph.x 的结构是 [batch_size, sequence_length * 2 + 1]
    其中前 sequence_length * 2 是 (x_t-k, y_t-k, ..., x_t, y_t)
    
    如果不修改 Dataset 类，我们需要在这里实时计算差分。
    """
    # 剥离时间特征（假设它是最后一个维度）
    time_feature = graph_x[:, -1:]
    pos_features = graph_x[:, :-1]
    
    # 重塑为 [num_nodes, sequence_length, 2]
    # 注意：这里需要根据您 Dataset 中具体的 flatten 顺序来调整
    num_nodes = pos_features.size(0)
    pos_seq = pos_features.view(num_nodes, sequence_len, 2)
    
    # 计算速度 v_t = p_t - p_{t-1}
    # 对于第一个时间步，由于没有前一个时间步，我们可以补0或者复制第二个时间步的速度
    # 这里采用计算相邻差分：结果形状 [num_nodes, sequence_len - 1, 2]
    velocities = pos_seq[:, 1:, :] - pos_seq[:, :-1, :]
    
    # 为了保持输入维度一致（或者您可以根据需要填充），通常需要处理掉减少的一个时间步
    # 方法1：填充第一帧速度为0
    zeros = torch.zeros(num_nodes, 1, 2, device=graph_x.device)
    velocities_padded = torch.cat([zeros, velocities], dim=1)
    
    # 展平回 [num_nodes, sequence_length * 2]
    velocities_flat = velocities_padded.view(num_nodes, -1)
    
    # 拼回时间特征
    new_features = torch.cat([velocities_flat, time_feature], dim=1)
    
    return new_features

def inference(num_rollout_steps: int = 10, edge_method='knn', edge_k=6, start_frame_idx: int = 0):
    """
    Args:
        num_rollout_steps: 预测步数
        edge_method: 建边方法
        edge_k: KNN的k值
        start_frame_idx: 从第几帧开始预测 (默认为0, 即从原始数据的第0帧开始)
    """
    
    all_predictions = []
    all_gt_trajectories = []  
    all_initial_positions = []
    all_histories = [] # <--- 新增：用于保存历史轨迹

    # 假设 history 包含了 (sequence_length * 2) 个特征，即过去 sequence_length 帧的 (dx, dy) 或 (x, y)
    # 注意：这里需要你确认 sequence_length 的定义是否在全局变量中
    num_history_features = sequence_length * 2 
    
    # ===== 第一阶段：收集所有帧的batch数据 =====
    all_frame_batches_raw = [] # 保存原始的所有帧
    
    for batch_idx, batch_graph in enumerate(test_loader):
        try:
            batch_graph = transformer(batch_graph)
        except Exception:
            batch_graph = transformer(batch_graph.to(device)).to('cpu')
        
        batch_graph = batch_graph.to(device)
        all_frame_batches_raw.append(batch_graph)
    
    # ===== 根据 start_frame_idx 裁剪帧 =====
    if start_frame_idx >= len(all_frame_batches_raw):
        raise ValueError(f"start_frame_idx ({start_frame_idx}) 不能大于或等于总帧数 ({len(all_frame_batches_raw)})")
        
    # 这里的 all_frame_batches 就是裁剪后的序列，下标 0 对应原始的 start_frame_idx
    all_frame_batches = all_frame_batches_raw[start_frame_idx:]
    
    num_frames = len(all_frame_batches)
    first_batch = all_frame_batches[0] # 这是我们的起始帧（即原始数据的第 start_frame_idx 帧）
    num_graphs_per_batch = len(first_batch.ptr) - 1
    
    # ===== 关键修复：GT的起始位置应该是 graph.pos =====
    # 确保预测步数不会超过剩余的帧数
    effective_rollout_steps = min(num_rollout_steps, num_frames - 1)
    
    print(f"第二阶段：执行Rollout预测 (从原始第 {start_frame_idx} 帧开始，预测 {effective_rollout_steps} 步)...")
    
    for graph_idx in range(num_graphs_per_batch):
        
        initial_frame_batch = all_frame_batches[0] # 当前序列的第0帧
        start_node = initial_frame_batch.ptr[graph_idx].item()
        end_node = initial_frame_batch.ptr[graph_idx + 1].item()
        node_indices = torch.arange(start_node, end_node)
        
        # 初始位置 (当前时刻 t=0 的位置)
        initial_positions = initial_frame_batch.pos[node_indices].clone()
        all_initial_positions.append(initial_positions.cpu().detach().numpy())

        # ===== 提取历史轨迹 =====
        # 假设 x 的结构是 [history_features (seq_len*2), time, other_features...]
        # 历史特征通常是相对位移或绝对位置。这里假设存储的是绝对位置或者可以通过某种方式恢复。
        # 如果模型训练时输入的是 [pos_t-seq, pos_t-seq+1, ..., pos_t-1]，且展平为 2*seq_len
        
        current_x = initial_frame_batch.x[node_indices].clone()
        
        # 提取历史部分
        history_flat = current_x[:, :num_history_features] # (N, sequence_length * 2)
        
        # Reshape 成 (N, sequence_length, 2)
        # 注意：这里取决于数据处理时的 stacking 顺序，通常是 (x, y, x, y...) 或 (x..., y...)
        # 这里假设是交替存储 (x_t-k, y_t-k, x_t-k+1, y_t-k+1 ...)
        history_unflattened = history_flat.view(history_flat.shape[0], sequence_length, 2)
        
        # 将其转换为 (sequence_length, N, 2) 并转为 numpy
        # 此时的历史轨迹是 t-sequence_length 到 t-1
        history_trajectory = history_unflattened.permute(1, 0, 2).cpu().detach().numpy()
        
        # 我们把当前时刻 t=0 (initial_positions) 也拼接到历史末尾，形成完整的“过去+现在”线条
        # 这样画图时是连续的
        current_pos_np = initial_positions.cpu().detach().numpy() # (N, 2)
        current_pos_expanded = np.expand_dims(current_pos_np, axis=0) # (1, N, 2)
        
        # 完整的历史路径: t_{-seq} ... t_{-1}, t_{0}
        full_history = np.concatenate([history_trajectory, current_pos_expanded], axis=0)
        all_histories.append(full_history)

        # ===== 开始预测逻辑 =====
        sample_predictions = []
        sample_predictions.append(initial_positions.cpu().detach().numpy())
        
        # ===== 收集GT轨迹 =====
        sample_gt_trajectory = []
        
        # 首先添加初始帧
        sample_gt_trajectory.append(initial_positions.cpu().detach().numpy())
        
        # 然后添加后续帧
        for frame_idx in range(1, effective_rollout_steps + 1):
            if frame_idx < num_frames:
                frame_batch = all_frame_batches[frame_idx]
                start_node = frame_batch.ptr[graph_idx].item()
                end_node = frame_batch.ptr[graph_idx + 1].item()
                node_indices_frame = torch.arange(start_node, end_node)
                
                gt_pos = frame_batch.pos[node_indices_frame].cpu().detach().numpy()
                sample_gt_trajectory.append(gt_pos)
        
        # ===== 执行Rollout预测 =====
        current_sample_pos = initial_positions.clone()
        current_sample_x = initial_frame_batch.x[node_indices].clone()
        
        original_edge_attr = initial_frame_batch.edge_attr
        
        current_sample_edge_index, current_sample_edge_attr = build_edges_from_geometry(
            pos=current_sample_pos,
            original_edge_attr=original_edge_attr,
            method=edge_method,
            k=edge_k,
            self_loop=False,
            device=device
        )

        # Rollout循环
        for step in range(effective_rollout_steps):
            with torch.no_grad():
                temp_graph = Data(
                    x=compute_velocity_features(current_sample_x, sequence_length),
                    pos=current_sample_pos,
                    edge_index=current_sample_edge_index,
                    edge_attr=current_sample_edge_attr
                )
                
                predicted_displacement_normalized = model(temp_graph)

                if isinstance(predicted_displacement_normalized, (tuple, list)):
                    predicted_displacement_normalized = predicted_displacement_normalized[0]
                if predicted_displacement_normalized.shape != current_sample_pos.shape:
                    predicted_displacement_normalized = predicted_displacement_normalized.view_as(current_sample_pos)

                # ✅ 反归一化
                predicted_displacement = simulator._output_normalizer.inverse(
                    predicted_displacement_normalized
                )
            
            # 更新位置
            predicted_positions = current_sample_pos + predicted_displacement
            sample_predictions.append(predicted_positions.cpu().detach().numpy())

            # 更新历史特征
            history = current_sample_x[:, :num_history_features]
            history_shifted = history[:, 2:]
            new_frame = predicted_positions
            updated_history = torch.cat([history_shifted, new_frame], dim=1)

            time_col = current_sample_x[:, num_history_features:num_history_features+1]
            new_time_col = time_col + time_step

            current_sample_x = torch.cat([updated_history, new_time_col], dim=1)
            current_sample_pos = predicted_positions

            # 重新构建边
            current_sample_edge_index, current_sample_edge_attr = build_edges_from_geometry(
                pos=current_sample_pos,
                original_edge_attr=original_edge_attr,
                method=edge_method,
                k=edge_k,
                self_loop=False,
                device=device
            )
        
        # 保存预测和GT
        all_predictions.append(np.stack(sample_predictions, axis=0))
        all_gt_trajectories.append(np.stack(sample_gt_trajectory, axis=0))
    
    # 返回增加了 all_histories
    return all_initial_positions, all_predictions, all_gt_trajectories, all_histories

In [4]:
def visualize_rollout_animation(
    initial_pos_list,
    predictions_list,
    gt_trajectory_list=None,
    batch_idx=0,
    interval=100,
    save_path=None,
    max_display=296,
    num_tracked_particles=5,
):
    """
    Visualize rollout results animation with frame-by-frame GT comparison,
    and plot error vs step (several particles + mean error).

    Args:
        initial_pos_list: list of (N,2) arrays - initial positions
        predictions_list: list of arrays, each shape (num_steps, N, 2) - predicted trajectories
        gt_trajectory_list: list of arrays, each shape (num_steps, N, 2) - GT trajectories
        batch_idx:  batch index to display
        interval: frame interval in milliseconds
        save_path:  save path for gif if specified
        max_display: max number of particles to display
        num_tracked_particles: how many individual particles to track in error plot
    """
    if batch_idx >= len(predictions_list):
        raise IndexError(f"batch_idx {batch_idx} exceeds list length")

    pred_trajectory = predictions_list[batch_idx]  # (T, N, 2)

    if pred_trajectory.ndim != 3:
        raise ValueError(f"Predicted trajectory should be 3D, got {pred_trajectory.ndim}D")

    num_steps, num_particles, _ = pred_trajectory.shape

    # -------- GT 处理 --------
    has_gt = False
    gt_trajectory = None

    if gt_trajectory_list is not None and batch_idx < len(gt_trajectory_list):
        gt_data = gt_trajectory_list[batch_idx]

        if gt_data is not None:
            gt_trajectory = np.array(gt_data)
            has_gt = True

            if gt_trajectory.ndim != 3:
                raise ValueError(f"GT trajectory should be 3D, got {gt_trajectory. ndim}D")

            if gt_trajectory.shape[0] != num_steps:
                raise ValueError(
                    f"GT trajectory steps {gt_trajectory.shape[0]} mismatch with prediction {num_steps}"
                )

    display_n = min(num_particles, max_display)

    # -------- 误差统计（只有有 GT 才算）--------
    if has_gt: 
        # (T, display_n, 2)
        disp_all = pred_trajectory[:, :display_n, : ] - gt_trajectory[:, :display_n, :]
        # (T, display_n)
        per_particle_error = np.linalg.norm(disp_all, axis=-1)
        # (T,)
        mean_error = per_particle_error.mean(axis=1)

        tracked_n = min(num_tracked_particles, display_n)
        tracked_indices = np.arange(tracked_n)
    else:
        per_particle_error = None
        mean_error = None
        tracked_indices = []

    # -------- 轨迹动画图像 --------
    fig, ax_main = plt.subplots(1, 1, figsize=(14, 10))

    # 坐标范围
    all_x = pred_trajectory[:, :display_n, 0]. flatten()
    all_y = pred_trajectory[:, :display_n, 1].flatten()

    if has_gt:
        all_x = np.concatenate([all_x, gt_trajectory[:, :display_n, 0].flatten()])
        all_y = np. concatenate([all_y, gt_trajectory[:, :display_n, 1].flatten()])

    if all_x.size == 0 or all_y.size == 0:
        raise RuntimeError("No data to visualize.")

    margin = 0.1 * max(
        all_x.max() - all_x.min(),
        all_y.max() - all_y.min(),
        1.0
    )

    # 主图设置
    ax_main.set_xlim(all_x.min() - margin, all_x.max() + margin)
    ax_main.set_ylim(all_y. min() - margin, all_y.max() + margin)
    ax_main.set_xlabel('X Position', fontsize=12)
    ax_main.set_ylabel('Y Position', fontsize=12)
    ax_main.set_aspect('equal')
    title = f'Predicted Trajectory - Batch {batch_idx}'
    if has_gt:
        title += ' (Blue:  Predicted, Red: Ground Truth)'
    ax_main.set_title(title, fontsize=14)

    # 预测粒子 + 轨迹
    pred_particles = ax_main.scatter(
        [], [], c='blue', s=60, alpha=0.8,
        label='Predicted', edgecolors='navy', linewidths=0.8
    )
    pred_trails = [
        ax_main.plot([], [], 'b-', alpha=0.3, linewidth=1.2)[0]
        for _ in range(display_n)
    ]

    # GT 粒子 + 轨迹
    if has_gt:
        gt_particles = ax_main.scatter(
            [], [], c='red', s=60, alpha=0.8,
            label='Ground Truth', marker='o',
            edgecolors='darkred', linewidths=0.8
        )
        gt_trails = [
            ax_main.plot([], [], 'r-', alpha=0.3, linewidth=1.2)[0]
            for _ in range(display_n)
        ]

        error_text = ax_main.text(
            0.98, 0.02, '',
            transform=ax_main.transAxes, fontsize=10,
            verticalalignment='bottom', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9, pad=0.8),
            family='monospace'
        )

    step_text = ax_main.text(
        0.02, 0.98, '',
        transform=ax_main.transAxes, fontsize=10,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.9, pad=0.8),
        family='monospace'
    )

    ax_main.legend(loc='upper right', fontsize=10, framealpha=0.95)
    ax_main.grid(True, alpha=0.3)

    # -------- 新增：误差曲线图 --------
    fig_err = None
    if has_gt: 
        fig_err, ax_err = plt.subplots(1, 1, figsize=(8, 5))

        steps = np.arange(num_steps)

        # 平均误差
        ax_err.plot(
            steps, mean_error,
            'k-', lw=2, label='Mean error'
        )

        # 若干粒子误差
        colors = plt.cm.tab10(np.linspace(0, 1, len(tracked_indices)))
        for idx, c in zip(tracked_indices, colors):
            ax_err.plot(
                steps, per_particle_error[:, idx],
                color=c, lw=1.2, alpha=0.8,
                label=f'Particle {idx}'
            )

        ax_err.set_xlabel('Step', fontsize=12)
        ax_err.set_ylabel('Displacement error (L2)', fontsize=12)
        ax_err.set_title(f'Prediction Error vs Step - Batch {batch_idx}', fontsize=14)
        ax_err.grid(True, alpha=0.3)
        ax_err.legend(fontsize=9)
        fig_err. tight_layout()

    # -------- 动画初始化 / 更新 --------
    def init():
        pred_particles.set_offsets(np.empty((0, 2)))
        for t in pred_trails:
            t.set_data([], [])
        if has_gt:
            gt_particles.set_offsets(np.empty((0, 2)))
            for t in gt_trails:
                t.set_data([], [])
            error_text.set_text('')
        step_text.set_text('')

        artists = [pred_particles] + pred_trails + [step_text]
        if has_gt:
            artists += [gt_particles] + gt_trails + [error_text]
        return artists

    def update(frame):
        # 预测位置
        pred_positions = pred_trajectory[frame, :display_n, :]
        pred_particles.set_offsets(pred_positions)

        progress = frame / max(1, (num_steps - 1))
        pred_particles.set_color(plt.cm.Blues(0.4 + 0.5 * progress))

        # 预测轨迹
        if frame > 0:
            for i in range(display_n):
                seg = pred_trajectory[: frame + 1, i, :]
                pred_trails[i].set_data(seg[:, 0], seg[:, 1])

        if has_gt:
            # GT 位置
            gt_positions = gt_trajectory[frame, : display_n, :]
            gt_particles.set_offsets(gt_positions)
            gt_particles.set_color(plt.cm.Reds(0.4 + 0.5 * progress))

            # GT 轨迹
            if frame > 0:
                for i in range(display_n):
                    seg = gt_trajectory[:frame + 1, i, :]
                    gt_trails[i].set_data(seg[:, 0], seg[:, 1])

            # 误差统计文本（当前帧）
            displacement = pred_positions - gt_positions
            mse = np.mean(displacement ** 2)
            mae = np.mean(np.abs(displacement))
            distances = np.linalg.norm(displacement, axis=1)

            error_text.set_text(
                f'Frame {frame}/{num_steps - 1}\n'
                f'─────────────────\n'
                f'MSE:        {mse:.6f}\n'
                f'MAE:       {mae:.6f}\n'
                f'Mean Dist: {np.mean(distances):.6f}\n'
                f'Max Dist:  {np.max(distances):.6f}'
            )

        step_text.set_text(
            f'Progress: {progress * 100:.1f}%\n'
            f'Particles: {display_n}'
        )

        artists = [pred_particles] + pred_trails + [step_text]
        if has_gt: 
            artists += [gt_particles] + gt_trails + [error_text]
        return artists

    anim = FuncAnimation(
        fig, update, init_func=init,
        frames=num_steps, interval=interval,
        blit=True, repeat=True
    )

    # -------- 保存动画 & 误差图 --------
    if save_path:
        try:
            save_dir = os.path.dirname(save_path)
            if save_dir and not os.path. exists(save_dir):
                os.makedirs(save_dir, exist_ok=True)

            print(f"[INFO] Saving animation to {save_path}...")
            writer = PillowWriter(fps=max(1, 1000 // interval))
            anim.save(save_path, writer=writer, dpi=100)
            print("[SUCCESS] Animation saved!")

            if has_gt and fig_err is not None:
                base, ext = os.path.splitext(save_path)
                error_fig_path = base + "_error.png"
                print(f"[INFO] Saving error figure to {error_fig_path}...")
                fig_err.savefig(error_fig_path, dpi=150)
                print("[SUCCESS] Error figure saved!")

        except Exception as e:
            print(f"[ERROR] Save failed: {e}")

    # -------- 新增：打印 GT 每帧平均运动距离 --------
    if has_gt:
        # 对所有粒子（不是 display_n，而是整个 N）统计更完整
        gt_pos = gt_trajectory  # (T, N, 2)
        gt_disp = gt_pos[1: ] - gt_pos[:-1]  # (T-1, N, 2)
        gt_step_dist = np.linalg.norm(gt_disp, axis=-1).mean(axis=1)  # (T-1,)

        print(f"[GT MOTION] Batch {batch_idx} per-frame mean displacement:")
        for t, d in enumerate(gt_step_dist, start=1):
            print(f"  Frame {t}: mean displacement = {d:.6f}")

    plt.tight_layout()
    plt.show()
    return anim


def compute_rollout_errors(predictions_list, ground_truth_list=None):
    """
    计算每个 rollout 时间步的平均预测误差（在所有 batch 和所有粒子上求平均）
    返回：numpy array, length = num_steps-1，每个元素是该时间步的 mean displacement magnitude
    """
    max_steps = max(pred.shape[0] for pred in predictions_list)
    step_errors = []  # 每个 step 收集所有 batch 的节点位移误差（或位移幅值）
    for step in range(1, max_steps):
        vals = []
        for batch_pred in predictions_list:
            if batch_pred.shape[0] <= step:
                continue
            disp = batch_pred[step] - batch_pred[step - 1]  # (N,2)
            mag = np.linalg.norm(disp, axis=1)  # (N,)
            vals.append(mag. mean())
        if len(vals) > 0:
            step_errors.append(np.mean(vals))
        else:
            step_errors. append(np.nan)
    return np.array(step_errors)

In [5]:
def visualize_rollout_with_history(
    initial_pos_list,
    predictions_list,
    gt_trajectory_list,
    history_frames_list,  # 新增：历史帧列表
    batch_idx=0,
    interval=150,
    save_path=None,
    max_display=200,
    history_steps_to_show=10  # 显示多少步历史
):
    """
    可视化包含历史轨迹、预测轨迹和真实未来轨迹的动画。
    
    Args:
        initial_pos_list: 初始位置列表
        predictions_list: 预测轨迹列表 (T_pred, N, 2)
        gt_trajectory_list: 真实未来轨迹列表 (T_pred, N, 2)
        history_frames_list: 历史帧数据列表 (T_hist, N, 2)
        batch_idx: 要可视化的 batch 索引
        interval: 动画帧间隔 (ms)
        history_steps_to_show: 即使历史数据很长，也只显示最近的多少帧
    """
    
    # 1. 数据准备与校验
    if batch_idx >= len(predictions_list):
        raise IndexError(f"batch_idx {batch_idx} exceeds list length")
        
    pred_data = predictions_list[batch_idx]  # (T_pred, N, 2)
    gt_data = gt_trajectory_list[batch_idx] if gt_trajectory_list else None # (T_pred, N, 2)
    hist_data = history_frames_list[batch_idx] if history_frames_list else None # (T_total_hist, N, 2)
    
    if hist_data is None:
        print("[WARNING] No history data provided. Creating dummy history.")
        hist_data = pred_data[0:1] # Fallback
    
    # 截取最近的历史帧
    T_hist_total = hist_data.shape[0]
    start_hist_idx = max(0, T_hist_total - history_steps_to_show)
    effective_hist_data = hist_data[start_hist_idx:]
    
    T_hist = effective_hist_data.shape[0]
    T_pred = pred_data.shape[0]
    N = pred_data.shape[1]
    
    display_n = min(N, max_display)
    
    # 2. 图像设置
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # 计算坐标范围 (考虑所有阶段的数据)
    all_x = np.concatenate([
        effective_hist_data[:, :display_n, 0].flatten(),
        pred_data[:, :display_n, 0].flatten()
    ])
    all_y = np.concatenate([
        effective_hist_data[:, :display_n, 1].flatten(),
        pred_data[:, :display_n, 1].flatten()
    ])
    
    if gt_data is not None:
        all_x = np.concatenate([all_x, gt_data[:, :display_n, 0].flatten()])
        all_y = np.concatenate([all_y, gt_data[:, :display_n, 1].flatten()])
        
    margin = 0.1 * max(all_x.max() - all_x.min(), all_y.max() - all_y.min(), 1.0)
    ax.set_xlim(all_x.min() - margin, all_x.max() + margin)
    ax.set_ylim(all_y.min() - margin, all_y.max() + margin)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3, linestyle='--')
    
    ax.set_title(f'History + Prediction vs GT (Batch {batch_idx})', fontsize=14)
    ax.set_xlabel('X Position')
    ax.set_ylabel('Y Position')

    # 3. 初始化绘图元素
    # A. 历史部分 (红色空心)
    hist_particles = ax.scatter([], [], c='none', s=60, alpha=0.8, label='History', marker='x')
    hist_trails = [ax.plot([], [], 'r--', alpha=0.4, linewidth=1.2)[0] for _ in range(display_n)]
    
    # B. 预测部分 (蓝色)
    pred_particles = ax.scatter([], [], c='blue', s=60, alpha=0.9, label='Prediction', edgecolors='white')
    pred_trails = [ax.plot([], [], 'b-', alpha=0.5, linewidth=1.5)[0] for _ in range(display_n)]
    
    # C. GT部分 (红色空心)
    gt_particles = ax.scatter([], [], c='none', s=60, alpha=0.8, label='Ground Truth', edgecolors='red', linewidth=1.5)
    gt_trails = [ax.plot([], [], 'r--', alpha=0.4, linewidth=1.2)[0] for _ in range(display_n)]
    
    # D. 状态指示器
    status_text = ax.text(0.02, 0.98, '', transform=ax.transAxes, verticalalignment='top',
                          bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    
    # 这里的 total_frames = 历史播放过程 + 预测播放过程
    # 为了动画连贯，我们在播放预测时，历史轨迹保持显示
    total_frames = T_hist + T_pred 
    
    ax.legend(loc='upper right')

    def init():
        hist_particles.set_offsets(np.empty((0, 2)))
        pred_particles.set_offsets(np.empty((0, 2)))
        gt_particles.set_offsets(np.empty((0, 2)))
        for lines in [hist_trails, pred_trails, gt_trails]:
            for line in lines:
                line.set_data([], [])
        status_text.set_text('')
        return [hist_particles, pred_particles, gt_particles, status_text] + hist_trails + pred_trails + gt_trails

    def update(frame_idx):
        artists = [status_text]
        
        # === 阶段 1: 播放历史 ===
        if frame_idx < T_hist:
            current_hist_pos = effective_hist_data[frame_idx, :display_n, :]
            hist_particles.set_offsets(current_hist_pos)
            
            # 绘制历史轨迹 (从开始到当前帧)
            if frame_idx > 0:
                for i in range(display_n):
                    seg = effective_hist_data[:frame_idx+1, i, :]
                    hist_trails[i].set_data(seg[:, 0], seg[:, 1])
            
            status_text.set_text(f'PHASE: History Playback\nFrame: -{T_hist - frame_idx}')
            artists += [hist_particles] + hist_trails

        # === 阶段 2: 播放预测与GT ===
        else:
            # 相对预测阶段的索引
            pred_step = frame_idx - T_hist
            
            # 1. 历史轨迹保持完全显示 (静态背景)
            hist_particles.set_offsets(effective_hist_data[-1, :display_n, :]) # 停在历史最后一帧位置
            for i in range(display_n):
                hist_trails[i].set_data(effective_hist_data[:, i, 0], effective_hist_data[:, i, 1])
            
            # 2. 更新预测
            current_pred_pos = pred_data[pred_step, :display_n, :]
            pred_particles.set_offsets(current_pred_pos)
            
            # 预测轨迹需要连接历史的最后一帧，以保证连续性
            # 构造连接点：[History_Last_Frame, Pred_Frame_0, Pred_Frame_1 ...]
            start_pos = effective_hist_data[-1:, :display_n, :] # (1, N, 2)
            
            # 绘制当前预测轨迹
            for i in range(display_n):
                # 拼接历史最后一帧和当前预测的所有帧
                current_path_x = np.concatenate([start_pos[0, i, 0:1], pred_data[:pred_step+1, i, 0]])
                current_path_y = np.concatenate([start_pos[0, i, 1:2], pred_data[:pred_step+1, i, 1]])
                pred_trails[i].set_data(current_path_x, current_path_y)
            
            artists += [pred_particles, hist_particles] + pred_trails + hist_trails

            # 3. 更新GT (如果存在)
            if gt_data is not None:
                current_gt_pos = gt_data[pred_step, :display_n, :]
                gt_particles.set_offsets(current_gt_pos)
                
                for i in range(display_n):
                    # 同样拼接历史最后一帧
                    current_gt_path_x = np.concatenate([start_pos[0, i, 0:1], gt_data[:pred_step+1, i, 0]])
                    current_gt_path_y = np.concatenate([start_pos[0, i, 1:2], gt_data[:pred_step+1, i, 1]])
                    gt_trails[i].set_data(current_gt_path_x, current_gt_path_y)
                
                # 计算当前帧误差
                err = np.linalg.norm(current_pred_pos - current_gt_pos, axis=1).mean()
                status_text.set_text(f'PHASE: Prediction vs GT\nStep: {pred_step}/{T_pred-1}\nMean Error: {err:.4f}')
                artists += [gt_particles] + gt_trails
            else:
                status_text.set_text(f'PHASE: Prediction\nStep: {pred_step}/{T_pred-1}')

        return artists

    anim = FuncAnimation(
        fig, update, init_func=init,
        frames=total_frames, interval=interval,
        blit=True, repeat=True
    )

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        print(f"Saving history animation to {save_path}...")
        anim.save(save_path, writer=PillowWriter(fps=1000//interval))
        print("Done.")
        
    plt.tight_layout()
    plt.show()
    return anim

In [ ]:
if __name__ == "__main__":
    # 加载数据集与 DataLoader
    dataset_cell = CellMotionRollout(dataset_dir=dataset_dir, sequence_length=sequence_length, split='test_e0n300')
    #train-Copy1 train test test_n800 valid valid_1
    test_loader = DataLoader(dataset=dataset_cell, batch_size=batch_size, num_workers=0)

    initial_positions, predictions, ground_truth, history_frames= inference(
    num_rollout_steps=max_rollout_steps, 
    edge_method='delaunay',
    start_frame_idx=100  # 这里设置从第xx帧开始
    )

    history_window = 10
    # 裁剪GT轨迹到与预测相同的帧数
    predictions_frames = predictions[0].shape[0]  
    ground_truth_cropped = [gt[:predictions_frames] for gt in ground_truth]

    # 然后用裁剪后的GT进行可视化
    for batch_idx in range(min(3, len(predictions))):
        save_path = f'gifs/rollout_animation_with_gt_batch_{batch_idx}.gif'
        anim = visualize_rollout_animation(
            initial_positions,
            predictions,
            gt_trajectory_list=ground_truth_cropped,  # 使用裁剪后的GT ground_truth_cropped
            batch_idx=batch_idx,
            interval=200,
            save_path=save_path,
            max_display=2000,
            num_tracked_particles=0,#
        )

#    visualize_rollout_with_history(
#    initial_pos_list=initial_positions,
#    predictions_list=predictions,
#    gt_trajectory_list=ground_truth_cropped,
#    history_frames_list=history_frames, # 传入上面准备好的历史列表
#    batch_idx=0,
#    interval=200,
#    save_path='gifs/history_rollout.gif',
#    history_steps_to_show= history_window
#)
    # 计算并打印位移误差随时间的变化
    errors = compute_rollout_errors(predictions, ground_truth_cropped)  # 使用裁剪后的GT

    # 计算并打印位移误差随时间的变化
    errors = compute_rollout_errors(predictions, ground_truth)
    print(f"Per-step average displacement (mean over batches and particles): {errors}")
    print(f"Average displacement per step (mean of steps): {np.nanmean(errors):.6f}")
    print(f"Std of step displacements: {np.nanstd(errors):.6f}")

第二阶段：执行Rollout预测 (从原始第 100 帧开始，预测 100 步)...
[INFO] Saving animation to gifs/rollout_animation_with_gt_batch_0.gif...
